In [12]:
import os
import numpy as np
import networkx as nx

from src.OrganoidMesh import OrganoidMesh
from src.nuclei_to_mesh_projection import *

### Setup paths to meshes and nuclei tables

In [13]:
data_dir = '../NicoleData/20250929/fractal_output'

# --- path to your cell nucleus table ---
CELLS_CSV = "../NicoleData/features_csv/features/cell_types_class.csv"   # <-- adjust

timepoint = "day4p5"
zarr_name = "r0.zarr"
well = "A06"
round_name = "0_fused_zillum_registered"
organoid_id = 31 #19 # example


mesh_path = f"{data_dir}/{timepoint}/{zarr_name}/{well[0]}/{well[1:]}/{round_name}/meshes/nnorg_linked_multi_annotated_class/{organoid_id}.vtp"
cells_df = pd.read_csv(CELLS_CSV)


### Project binary markers to mesh

In [14]:
# Extract nuclei data matching the mesh label
organoid_id = os.path.splitext(os.path.basename(mesh_path))[0]
label_uid = f"{timepoint}_{well}_{organoid_id}"

nuclei_df_org = cells_df[cells_df["label_uid"] == label_uid].copy()

# Load membrane mesh as an OrganoidMesh
mesh = OrganoidMesh(mesh_path)

# Extract per-cell attributes from sub-table
nuclei_xyz, markers_bin = extract_cell_attributes(nuclei_df_org)

# Rescale coordinates and filter expression
_, nuclei_xyz = center_and_rescale_mesh(mesh, nuclei_xyz)
markers_bin = filter_lgr5_coexpression(markers_bin) 

# Project nuclei -> mesh vertices (using geometry from the mesh object)
proj_vertex_ids, proj_points = project_nuclei_to_mesh(
    nuclei_xyz,
    mesh,
    resolve_duplicates=True, # if multiple nuclei are projected to the same vertex, shift them slightly
)

# Compute Laplace-Beltrami eigen-decomposition for geodesics 
mesh._eig_decomp()

# Geodesic distances + Voronoi assignment
dist_mat, vertex_owner = compute_geodesic_voronoi(mesh, proj_vertex_ids)

# Assign cell quantities to mesh
marker_fields = assign_cell_quantity_to_vertices(vertex_owner, markers_bin)

print(markers_bin.shape)
print(marker_fields.shape)

Duplicate projections: 9 vertices with duplicates, 9 duplicate cell pairs.


heat sources: 100%|██████████| 1030/1030 [00:04<00:00, 233.69it/s]


(1030, 10)
(18001, 10)


In [15]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

marker_names = ['LGR5', 'Chromogranin A', 'Cyclin D', 'Mucin 2', 'AldoB',
                'Glucagon', 'Cyclin A', 'Agr2', 'Serotonin', 'Lysozyme']

# --- Markers to plot ---
marker_list = ["Serotonin", "Lysozyme", "LGR5"]

# --- Get their indices ---
marker_indices = [marker_names.index(name) for name in marker_list]

# --- Create subplot figure ---
fig = make_subplots(
    rows=1, cols=len(marker_list),
    specs=[[{'type': 'mesh3d'}] * len(marker_list)],
    subplot_titles=marker_list
)

# --- Add each marker as a Mesh3d trace ---
for i, (name, idx) in enumerate(zip(marker_list, marker_indices), start=1):
    fig.add_trace(
        go.Mesh3d(
            x=mesh.v[:, 0], y=mesh.v[:, 1], z=mesh.v[:, 2],
            i=mesh.f[:, 0], j=mesh.f[:, 1], k=mesh.f[:, 2],
            intensity=marker_fields[:, idx],
            colorscale='Viridis',
            showscale=True,
            name=name
        ),
        row=1, col=i
    )

# --- Adjust layout ---
for j in range(len(marker_list)):
    fig.update_layout(**{f"scene{'' if j == 0 else j+1}": dict(aspectmode='data')})

fig.update_layout(
    title_text="Marker intensity maps: Serotonin, Lysozyme, LGR5",
    showlegend=False
)

fig.show()

In [16]:

# Build arrays for the connection segments
line_x = []
line_y = []
line_z = []

for p_nuc, p_proj in zip(nuclei_xyz, proj_points):
    line_x += [p_nuc[0], p_proj[0], None]
    line_y += [p_nuc[1], p_proj[1], None]
    line_z += [p_nuc[2], p_proj[2], None]

fig = go.Figure(data=[
    # Connection lines: nucleus -> projected point
    go.Scatter3d(
        x=line_x,
        y=line_y,
        z=line_z,
        mode="lines",
        line=dict(width=2, color="rgba(0,0,0,0.5)"),  # semi-transparent gray
        name="Projection lines"
    ),

    # Projected points on the surface (red)
    go.Scatter3d(
        x=proj_points[:, 0],
        y=proj_points[:, 1],
        z=proj_points[:, 2],
        mode='markers',
        marker=dict(size=5, color='red', symbol='circle'),
        name="Projected nuclei on surface"
    ),

    # True nucleus centroids in volume (blue)
    go.Scatter3d(
        x=nuclei_xyz[:, 0],
        y=nuclei_xyz[:, 1],
        z=nuclei_xyz[:, 2],
        mode='markers',
        marker=dict(size=5, color='blue', symbol='circle'),
        name="Nucleus centroids"
    )
])

fig.update_layout(
    scene=dict(aspectmode='data'),
    title="Projected nuclei on surface (red) vs cell centroids (blue) with projection lines"
)

fig.show()
